# D · Phase 2: LLM + examiner profile (Disposition, Remaining Rounds, Time to Next Action)

Protocol: `D_trajectory/personalization_protocol.md` (frozen before any run).

**Before you start**, My Drive must hold:
- the `D_trajectory` folder with the new files `code/profile_inject.py`, `code/compare_profile_runs.py`, the updated
  `code/run_trajectory_llm.py` and `code/run_rounds_next_llm.py`, and `prompts/*_profile.txt`;
- `data_v1_share_v1.2.zip` (as before);
- the team package `B_D_C_phase1_v4_reviewed.zip`: it sits in Huijing's shared folder, so right-click it in Drive →
  **Organize → Add shortcut → My Drive** (the notebook checks its sha256).

How to run:
1. **Step 0:** choose the models and the stage. First time: `STAGE = 'formatcheck'` (5 cases, about $0.30 for GPT).
2. Secrets (key icon, Notebook access on), same as before: `OPENAI_API_KEY` or the three `AZURE_OPENAI_*`; Qwen: `QWEN_BASE_URL`, `QWEN_AZURE_API_KEY`.
3. **Runtime → Run all.** If the format check passes, change to `STAGE = 'full'` and **Run all** again (the 5 checked cases come from the cache).

In [ ]:
# Step 0: choices
RUN_GPT = True      # GPT-5-mini, reasoning_effort low (about $3 per profile setting for both prompts)
RUN_QWEN = False    # Qwen3.6-27B team setting: needs the live Azure deployment (billed per hour). Delete it when done.
STAGE = 'formatcheck'   # 'formatcheck' = 5 fixed cases (protocol section 3), 'full' = all 100 test_core cases
SETTINGS = ['own', 'mismatched']   # art-unit is not run with the LLM (protocol section 2)
CHECK_IDS = '13060040,13484627,13627444,13577408,14031839'

In [ ]:
# Step 1: find the files on Drive, unzip the data and the profile package, check hashes, install the tools
from google.colab import drive
drive.mount('/content/drive')
import glob, hashlib, json, os, subprocess, sys
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips = glob.glob('/content/drive/MyDrive/**/data_v1_share_v1.2.zip', recursive=True)
pkgs = (glob.glob('/content/drive/MyDrive/**/B_D_C_phase1_v4_reviewed.zip', recursive=True) +
        glob.glob('/content/drive/.shortcut-targets-by-id/**/B_D_C_phase1_v4_reviewed.zip', recursive=True))
assert roots, 'Cannot find the D_trajectory folder in My Drive.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive.'
assert pkgs, 'Cannot find B_D_C_phase1_v4_reviewed.zip in My Drive. In Drive: right-click the team zip → Organize → Add shortcut → My Drive.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
need = ['code/profile_inject.py', 'code/compare_profile_runs.py', 'code/run_trajectory_llm.py', 'code/run_rounds_next_llm.py',
        'code/score_trajectory.py', 'code/score_rounds_next.py', 'code/traj_common.py',
        'prompts/trajectory_forecast_profile.txt', 'prompts/rounds_next_forecast_profile.txt',
        'inputs/state_predictions_A.jsonl', 'inputs/trajectory_inputs.jsonl', 'personalization_protocol.md',
        'outputs/D/trajectory_predictions_gpt-5-mini_state.jsonl', 'outputs/D/rounds_next_predictions_gpt-5-mini_state.jsonl',
        'outputs/D/trajectory_predictions_prior_own.jsonl', 'outputs/D/rounds_next_predictions_prior_own.jsonl']
missing = [n for n in need if not os.path.exists(os.path.join(ROOT, n))]
assert not missing, f'Missing in D_trajectory on Drive: {missing}'
assert 'add_profile_args' in open(os.path.join(ROOT, 'code', 'run_trajectory_llm.py')).read(), 'Old run_trajectory_llm.py on Drive: copy the new one.'

# frozen hashes (protocol section 1)
PKG_SHA256 = 'd1e6e358757ba758a17373f14894449284253da1cfe4d775a0bbac6a7b207d8f'
PROMPTS = {'trajectory_forecast_profile.txt': 'd7e75c380ac8', 'rounds_next_forecast_profile.txt': '8656a89b9b39',
           'trajectory_forecast.txt': '8fa99d848c11', 'rounds_next_forecast.txt': '2e8f78f097b3'}
got = hashlib.sha256(open(pkgs[0], 'rb').read()).hexdigest()
assert got == PKG_SHA256, f'Profile package is not the frozen v4 file (sha256 {got}).'
for f, h in PROMPTS.items():
    g = hashlib.sha1(open(os.path.join(ROOT, 'prompts', f), 'rb').read()).hexdigest()[:12]
    assert g == h, f'Prompt {f} changed ({g}, frozen {h}). Do not edit prompts after the protocol.'

DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
PKG = '/content/B_D_C_phase1_v4_reviewed'
if not os.path.exists(PKG + '/RELEASE_MANIFEST.json'):
    subprocess.run(['unzip', '-q', '-o', pkgs[0], '-d', '/content/'], check=True)
assert os.path.exists(PKG + '/RELEASE_MANIFEST.json'), 'Package unzipped to an unexpected folder.'
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
print('ROOT =', ROOT); print('DATA =', DATA); print('PKG  =', PKG)
!pip -q install "openai==2.54.0" "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

In [ ]:
# Step 2: read the secrets (never typed into the notebook) and test the connection
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
def sh(args):
    p = subprocess.Popen([sys.executable, '-u'] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end=''); out.append(line)
    p.wait()
    return p.returncode, ''.join(out)
if RUN_GPT:
    if get('AZURE_OPENAI_ENDPOINT') and get('AZURE_OPENAI_API_KEY'):   # same route as the no-profile baselines
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_DEPLOYMENT']:
            assert get(k), f'Secret {k} is missing'
            os.environ[k] = get(k)
        os.environ.pop('OPENAI_API_KEY', None)
        print('GPT route: Azure, deployment', os.environ['AZURE_OPENAI_DEPLOYMENT'])
    else:
        assert get('OPENAI_API_KEY'), 'Add the Azure secrets or OPENAI_API_KEY.'
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY']:
            os.environ.pop(k, None)
        os.environ['OPENAI_API_KEY'] = get('OPENAI_API_KEY'); os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5-mini'
        print('GPT route: OpenAI API, model gpt-5-mini (the baselines used Azure; note this in the report)')
    rc, out = sh(['run_trajectory_llm.py', '--provider', 'azure', '--root', ROOT, '--data', DATA, '--ping'])
    assert 'connection ok' in out, 'GPT test failed. Send the message above to Claude.'
if RUN_QWEN:
    for k in ['QWEN_BASE_URL', 'QWEN_AZURE_API_KEY']:
        assert get(k), f'Secret {k} is missing'
        os.environ[k] = get(k)
    os.environ['QWEN_DEPLOYMENT'] = get('QWEN_DEPLOYMENT') or 'qwen--qwen36-27b'
    rc, out = sh(['run_trajectory_llm.py', '--provider', 'qwen', '--root', ROOT, '--data', DATA, '--ping'])
    assert 'connection ok' in out, 'Qwen test failed. Send the message above to Claude.'
print('Step 2 OK')

In [ ]:
# Step 3: run. Same state input (A_v1) as the no-profile baselines; only the profile is added.
def run(script, provider, setting, extra):
    args = [script, '--provider', provider, '--root', ROOT, '--data', DATA, '--split', 'core',
            '--state', STATE, '--state-version', 'A_v1', '--profile-pkg', PKG, '--profile-setting', setting] + extra
    return sh(args)
jobs = [p for p, ok in (('azure', RUN_GPT), ('qwen', RUN_QWEN)) if ok]
for provider in jobs:
    for setting in SETTINGS:
        for script in ['run_trajectory_llm.py', 'run_rounds_next_llm.py']:
            if STAGE == 'formatcheck':
                tag = ('qwen' if provider == 'qwen' else 'gpt') + '_' + setting + '_' + script.split('_')[1]
                extra = ['--ids', CHECK_IDS, '--workers', '1',
                         '--dump-prompts', os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_' + tag)]
            else:
                extra = ['--workers', '2']
            rc, out = run(script, provider, setting, extra)
            assert rc in (0, None), f'{provider} {setting} {script} stopped. Send the message above to Claude.'
print('Step 3 OK')

In [ ]:
# Step 4 (format check only): the pass rules of protocol section 3. Accuracy is NOT looked at.
import pandas as pd
if STAGE == 'formatcheck':
    rows = []
    for f in sorted(glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', '*_profile_*_formatcheck.jsonl'))):
        for line in open(f):
            r = json.loads(line)
            rows.append(dict(file=os.path.basename(f).replace('_formatcheck.jsonl', ''), case=r['sample_id'], status=r['status'],
                             finish=','.join(r.get('finish_reasons') or []), attempts=r.get('n_attempts'),
                             k=r.get('retrieved_k'), prompt_tokens=r.get('prompt_tokens'), error=str(r.get('error') or '')[:80]))
    T = pd.DataFrame(rows); display(T)
    bad = T[(T.status != 'ok') | T.finish.str.contains('length')]
    # the profile block must sit before "Task" and carry the case's cutoff date
    for d in glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_*')):
        for p in glob.glob(d + '/*.txt'):
            t = open(p, encoding='utf-8').read()
            i, j = t.find('Examiner history reference available before'), t.find('\nTask')
            assert 0 < i < j, f'Profile block missing or misplaced in {p}'
            cut = t.split('cutoff date ', 1)[1][:10]
            assert 'available before ' + cut in t, f'Profile cutoff differs from the case cutoff in {p}'
    print('One prompt, around the profile block:')
    t = open(sorted(glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_*', '13577408.txt')))[0], encoding='utf-8').read()
    i = t.find('- Examiner history reference for this'); print(t[i:i + 1800])
    print('\nFORMAT CHECK', 'PASSED' if len(bad) == 0 and len(T) == 20 * len(jobs) else 'NOT PASSED: see the table')

In [ ]:
# Step 5 (full stage only): score everything and run the frozen comparisons
if STAGE == 'full':
    sh(['score_trajectory.py', '--root', ROOT])
    sh(['score_rounds_next.py', '--root', ROOT])
    sh(['compare_profile_runs.py', '--root', ROOT])
    print('ALL DONE. Take a screenshot of this page and send it to Claude. Delete the Qwen deployment now if you used it.')